# Joint APTOS + EyePACS training, cropped arm ("J-crop") — record §94

**An ablation of the joint model with one change:** every image is cut to the bounding box of the fundus before the
resize (the rule fixed in record §89, unchanged). Data parts, the 1 : 2 epoch, the seeded EyePACS rotation, class weights,
batch 16, optimiser, augmentation, the selection metric on both full validation sets and the fixed 0.5 decode are those of
the direct arm (`joint_aptos_eyepacs.ipynb`, record §93).

- The direct-resize joint model stays the pre-declared primary model whatever this arm shows.
- Three seeds, no continuation rule. **One seed per runtime:** set `SEED` in cell 2 (42, 123 or 2026); with three accounts
  run one seed on each. Never run the same seed from two runtimes.
- Not read: the EyePACS held-out test, IDRiD, DDR.

Disk: the cropped EyePACS frames (51.4 GiB) and cropped APTOS frames (5.7 GiB) are copied to `/content` — about 57 GiB.
Time on a T4: about 30 minutes of staging, then about 9 minutes per epoch; 5 – 5.5 hours expected per seed. Resumable: after a
disconnect, run the cells again with the same `SEED`.

**Limitations that travel with the result:** both validation sets are in-domain for this model and select its checkpoint,
so a difference from the direct arm on them says little about unseen data; batch 16 against batch 2 for P.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / Stage-4 weights needed; no external test set is used.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad, arch1_train as at
import e1_data as ed, e1_train as e1t, e2_control as e2
import eyepacs_adaptation_data as ea, eyepacs_adaptation_train as eat, ep_aptos_train as ep, joint_train as jt, crop_frames as cf
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] SWITCHES -- everything expensive is OFF. Set a flag to True (and type its phrase) to allow that step. ====
SEED       = 42      # 42, 123 or 2026 -- ONE seed per runtime
STAGE_DATA = False   # copy the cropped EyePACS frames (51.4 GiB) and cropped APTOS frames (5.7 GiB) to /content
RUN_SEED   = False   # train (or resume) this seed of the cropped arm, evaluate BEST, compare
CONFIRM = {"crop": ""}
PHRASES = {"crop": "TRAIN THE CROPPED JOINT MODEL"}

def confirmed(step, flag):
    if flag and CONFIRM[step] != PHRASES[step]:
        raise RuntimeError(f"{step}: the flag is True but CONFIRM['{step}'] is not the phrase {PHRASES[step]!r}")
    return bool(flag)
assert SEED in jt.SEEDS
print({"SEED": SEED, "STAGE_DATA": STAGE_DATA, "RUN_SEED": RUN_SEED})

In [ ]:
# ==== [3] INPUTS AND PROTOCOL (reads manifests and small files only; trains nothing) ====
EXP = colab_config.DRIVE.experiments_root
assert jt.PROTOCOL == dict(at.P_PROTOCOL, batch_size=16) == eat.PROTOCOL and jt.ARMS["crop"] == {"prefix": "jc", "continuation_rule": False}
assert cf.RULE == {"gray_threshold": 12, "line_fraction": 0.02, "min_side_fraction": 0.1}       # the rule of record 89
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
assert cache.sha256_file(PRETRAINED) == pl.WEIGHTS_SHA256
DRIVE_EYEPACS = posixpath.join(colab_config.DATASET_ROOT, "EyePACS", "adaptation_frames_crop_v1")
DRIVE_APTOS = posixpath.join(colab_config.DATASET_ROOT, "APTOS2019", "frames_crop_v1")
LOCAL_EYEPACS, LOCAL_APTOS = "/content/eyepacs_adaptation_frames_crop_v1", "/content/aptos_frames_crop_v1"
EYEPACS_PLAN = {s: [(i, g) for i, g, _ in v] for s, v in cf.eyepacs_plan("").items()}       # the pinned adaptation split, shard order
APTOS_PLAN = {s: [(i, g) for i, g, _ in v] for s, v in cf.aptos_plan("").items()}           # the authoritative APTOS split
_tr, _va, _split_sha = msr.verify_split()
_empty = set(v2cfg.APTOS_EMPTY_FOV_IDS)
assert _split_sha == v2cfg.SPLIT_SHA256
assert [i for i, _ in APTOS_PLAN["train"]] == [str(i) for i, _ in _tr if str(i) not in _empty] and len(APTOS_PLAN["train"]) == 2921
assert [i for i, _ in APTOS_PLAN["val"]] == [str(i) for i, _ in _va if str(i) not in _empty] and len(APTOS_PLAN["val"]) == 730
assert (len(EYEPACS_PLAN["train"]), len(EYEPACS_PLAN["val"])) == (31610, 3512)
P_TABLE = posixpath.join(EXP, "PL_ConvNeXtPriors", "2026-09-28_05-22-44", "P_convnext_rgb", f"seed_{SEED}", "evaluation", "per_sample_best.csv")
EYEPACS_ONLY_TABLE = posixpath.join(eat.run_dir_for(EXP, ea.MANIFEST_SHA256), "metrics", "per_sample_best.csv")
_J = posixpath.join(jt.run_dir_for(EXP, SEED, "direct"), "metrics")                         # the direct arm of the same seed, if it is done
REFERENCES = {"p_aptos": P_TABLE, "eyepacs_only": EYEPACS_ONLY_TABLE, "joint_aptos": posixpath.join(_J, "per_sample_best_aptos_val.csv"),
              "joint_eyepacs": posixpath.join(_J, "per_sample_best_eyepacs_val.csv")}
assert os.path.exists(P_TABLE) and os.path.exists(EYEPACS_ONLY_TABLE)
print("protocol", jt.PROTOCOL, "| selection:", jt.SELECTION)
print("crop rule", cf.RULE, "| direct arm of this seed available for comparison:", os.path.exists(REFERENCES["joint_aptos"]))
print("cropped arm:", {s: jt.state(jt.run_dir_for(EXP, s, "crop")) for s in jt.SEEDS}, "| commit", REPO_COMMIT)

In [ ]:
# ==== [4] DATA: Drive -> local disk (every shard checked by SHA-256), then the four parts and the weight check ====
PARTS = FRAMES = None
if STAGE_DATA:
    print(ea.stage_cache(DRIVE_EYEPACS, LOCAL_EYEPACS))
    print(ea.stage_cache(DRIVE_APTOS, LOCAL_APTOS))
if os.path.exists(posixpath.join(LOCAL_EYEPACS, "index.json")) and os.path.exists(posixpath.join(LOCAL_APTOS, "index.json")):
    EYEPACS_FRAMES = cf.CropFrames(LOCAL_EYEPACS, EYEPACS_PLAN, check_hashes=False)        # staging hashed every shard
    APTOS_FRAMES = cf.CropFrames(LOCAL_APTOS, APTOS_PLAN, check_hashes=False)
    PARTS = (APTOS_FRAMES.entries("train"), APTOS_FRAMES.entries("val"), EYEPACS_FRAMES.entries("train"), EYEPACS_FRAMES.entries("val"))
    jt.check_data(*PARTS)                                                                  # no validation image in a training part
    FRAMES = jt.JointFrames(APTOS_FRAMES, EYEPACS_FRAMES, [i for i, _ in PARTS[0] + PARTS[1]], [i for i, _ in PARTS[2] + PARTS[3]])
    IDENTITIES = {"aptos": {"split_sha256": v2cfg.SPLIT_SHA256, "fingerprint": APTOS_FRAMES.fingerprint},
                  "eyepacs": {"split_manifest_sha256": ea.MANIFEST_SHA256, "fingerprint": EYEPACS_FRAMES.fingerprint}}
    _expected, _weights, _sampled = jt.class_weights(PARTS[0], PARTS[2], SEED)
    print(f"seed {SEED}: weights {[round(w, 4) for w in _weights]} | sampled-mixture weights {[round(w, 4) for w in _sampled['weights']]}")
    print("APTOS train / val", len(PARTS[0]), len(PARTS[1]), "| EyePACS train / val", len(PARTS[2]), len(PARTS[3]))
else:
    print("data not staged (set STAGE_DATA = True once per runtime)")

In [ ]:
# ==== [5] THIS SEED OF THE CROPPED ARM -- TRAINS (about 5 - 5.5 h on a T4; resumable). Then BEST is evaluated and compared. ====
def run_joint(seed):
    tf.keras.backend.clear_session()
    _, _arrays = pl.load_reference(PRETRAINED)
    return jt.run_seed(seed, EXP, FRAMES, PARTS, _arrays, IDENTITIES, REFERENCES,
                       repo_dir=REPO_DIR, staging_root="/content/joint_staging", arm="crop")

def show(result):
    b, c = result["best"], result["comparisons"]
    print(f"seed {result['seed']}: BEST epoch index {b['checkpoint']['epoch_index']} of {result['epochs_scored']} scored"
          f" | selection metric {b['checkpoint']['selection_metric']:.4f} | stop: {result['stop']}")
    for _name, _key, _ref in (("aptos_val", "aptos_val_vs_p", "P (same seed)"), ("eyepacs_val", "eyepacs_val_vs_eyepacs_only", "EyePACS-only model")):
        _m, _d = b["sets"][_name], c[_key]
        print(f"  {_name}: QWK {_m['qwk']:.4f} | {_ref} {_d['reference_qwk']:.4f} | difference {_d['difference']:+.4f} [{_d['ci_95'][0]:+.4f}, {_d['ci_95'][1]:+.4f}]")
        print(f"     AUROC cuts { {k: round(v, 3) for k, v in _m['auroc_cuts'].items()} } | recall { [round(_m['recall_per_grade'][str(g)] if str(g) in _m['recall_per_grade'] else _m['recall_per_grade'][g], 3) for g in range(5)] }")
        print(f"     referable sens / spec {_m['referable_sensitivity']:.3f} / {_m['referable_specificity']:.3f} | ECE {_m['ece_top_label']:.3f}")
    if "continuation" in result:
        print("  continuation rule:", json.dumps(result["continuation"]))
    for _l in result["limitations"]:
        print("  limitation:", _l)

if confirmed("crop", RUN_SEED):
    assert FRAMES is not None, "stage the data first (cell 4)"
    RESULT = run_joint(SEED)
    show(RESULT)
    for _key in ("aptos_val_vs_direct_arm", "eyepacs_val_vs_direct_arm"):
        if _key in RESULT["comparisons"]:
            _d = RESULT["comparisons"][_key]
            print(f"  {_key}: cropped {_d['joint_qwk']:.4f} | direct {_d['reference_qwk']:.4f} | difference {_d['difference']:+.4f} [{_d['ci_95'][0]:+.4f}, {_d['ci_95'][1]:+.4f}]")
else:
    print(f"seed {SEED} of the cropped arm not run (RUN_SEED = False):", jt.state(jt.run_dir_for(EXP, SEED, "crop")))

In [ ]:
# ==== [6] PER-EPOCH VALIDATION HISTORY of this seed (reads what the run wrote) ====
_dir = posixpath.join(jt.run_dir_for(EXP, SEED, "crop"), "dual_validation")
if os.path.isdir(_dir):
    for _f in sorted(os.listdir(_dir)):
        _e = json.load(open(posixpath.join(_dir, _f)))
        print(f"epoch {_e['epoch']:>2}: APTOS {_e['val_QWK_aptos']:.4f} | EyePACS {_e['val_QWK_eyepacs']:.4f} | mean {_e['selection_metric']:.4f}")
else:
    print("no epoch has been scored yet")